# Stiffness 9 — dead volume in the flowsheet

A real system is not just a column. There is tubing between the injector and the inlet frit, and a detector cell after the outlet, and both hold liquid. Each is well mixed to a good approximation, so each is a stirred tank that relaxes at

$$\frac{1}{\tau} = \frac{Q}{V}.$$

**A small volume is a fast tank.** At this column's 2 mL/min, a 10 µL detector cell has $\tau = 0.3\,\mathrm{s}$ — faster than anything in the column except the grid. That is a fast process introduced by the *plumbing*, not by the chemistry, and it is the last place in this chapter where one might hide.

The flowsheet is

```
inlet  ->  column  ->  detector  ->  outlet
```

with the three protein separation in the column, and the knob is the detector volume. What is plotted and recorded is the **detector outlet** — the signal an instrument would actually show.

| for orientation | volume | τ |
|---|---|---|
| detector flow cell | 5–20 µL | 0.15–0.6 s |
| short connecting tube | 50–150 µL | 1.5–4.5 s |
| mixing chamber | 0.5–2 mL | 15–60 s |

In [ ]:
import utility.flowsheet_explorer as explorer

explorer.show_setting()

## What to watch

The sweep runs from a 10 mL mixing chamber down to 0.1 nL — a volume no plumbing could hold, included to push the time constant into the microseconds.

Two columns matter. **`rate / grid rate`** says when the tank has overtaken the fastest process already in the model. **`peak height`** says whether the tank is changing the answer: a large tank smears the chromatogram, a small one leaves it alone.

In [ ]:
from ipywidgets import interact

show_flowsheet = explorer.make_explorer()

interact(show_flowsheet, **show_flowsheet.sliders);

## What happened

**Nothing, again.**

| detector | τ | mixing rate | rate / grid | BDF steps | peak |
|---|---|---|---|---|---|
| 10 mL | 303 s | 0.0033 s⁻¹ | 0.012 | 1657 | **0.0024** |
| 100 µL | 3.0 s | 0.33 s⁻¹ | 1.2 | 1668 | 0.0059 |
| 10 µL | 0.30 s | 3.3 s⁻¹ | 12 | 1673 | 0.0059 |
| 0.1 nL | 3 µs | 3.3·10⁵ s⁻¹ | **1.2·10⁶** | **1672** | 0.0059 |

Eight decades of volume, a mixing rate that ends up more than a million times faster than convection across a column cell, and the step count moves by **0.9 %**. The mean BDF order is 5.000 on every row, and nothing fails even at a time constant of three microseconds.

**The volume that matters is the big one, and it matters for a different reason.** A 10 mL chamber cuts the peak height by more than half, from 0.0059 to 0.0024. That is the tank smearing the separation you just performed — a real effect on the answer, which is why dead volume is worth minimising in an instrument. It costs nothing in compute. Below about 100 µL the tank has stopped affecting the chromatogram altogether, and everything smaller is free in both senses.

So the plumbing behaves exactly like the binding, the reaction and the particle: **a fast, well damped mode is carried rather than resolved.** The mechanism does not care that this one came from a piece of tubing instead of from chemistry.

> **Take-away.** You never need to simplify a flowsheet for the sake of the time integrator. Small dead volumes are free, however small — and large ones are a modelling decision about your instrument, not a performance one. If you are tempted to drop a detector cell or a connecting tube from a model to make it faster, that is not where the time is going.